# Dataset Download

In [ ]:
!wget -r -N -c -np https://physionet.org/files/gaitpdb/1.0.0/

--2025-03-24 21:45:12--  https://physionet.org/files/gaitpdb/1.0.0/
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘physionet.org/files/gaitpdb/1.0.0/index.html’

physionet.org/files     [ <=>                ]  36.17K  --.-KB/s    in 0.05s   

Last-modified header missing -- time-stamps turned off.
2025-03-24 21:45:12 (784 KB/s) - ‘physionet.org/files/gaitpdb/1.0.0/index.html’ saved [37036]

Loading robots.txt; please ignore errors.
--2025-03-24 21:45:12--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 22 [text/plain]
Saving to: ‘physionet.org/robots.txt’

physionet.org/robot 100%[===================>]      22  --.-KB/s    in 0s      

2025-03-24 21:45:12 (15.4 MB/s) - ‘physionet.org/robots.txt’ saved [22/22]

--2025-03-24 21

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -la /content/physionet.org/files/gaitpdb/1.0.0/

total 295992
drwxr-xr-x 2 root root   12288 Mar 24 21:57 .
drwxr-xr-x 3 root root    4096 Mar 24 21:45 ..
-rw-r--r-- 1 root root  106031 Feb 25  2008 demographics.html
-rw-r--r-- 1 root root   15665 Jun  4  2006 demographics.txt
-rw-r--r-- 1 root root   47104 Feb 25  2008 demographics.xls
-rw-r--r-- 1 root root    2339 Feb 25  2008 format.txt
-rw-r--r-- 1 root root 1116032 Nov 15  2004 GaCo01_01.txt
-rw-r--r-- 1 root root 1058168 Nov 15  2004 GaCo02_01.txt
-rw-r--r-- 1 root root 1061318 Nov 15  2004 GaCo02_02.txt
-rw-r--r-- 1 root root 1135895 Nov 15  2004 GaCo03_01.txt
-rw-r--r-- 1 root root 1127589 Nov 15  2004 GaCo03_02.txt
-rw-r--r-- 1 root root 1070333 Nov 15  2004 GaCo04_01.txt
-rw-r--r-- 1 root root 1069887 Nov 15  2004 GaCo04_02.txt
-rw-r--r-- 1 root root 1131521 Nov 15  2004 GaCo05_01.txt
-rw-r--r-- 1 root root 1131186 Nov 15  2004 GaCo05_02.txt
-rw-r--r-- 1 root root 1075667 Nov 15  2004 GaCo06_01.txt
-rw-r--r-- 1 root root 1029132 Nov 15  2004 GaCo06_02.txt
-rw-r--r-- 1 root

In [ ]:
!cp -r /content/physionet.org/files/gaitpdb/1.0.0/* "/content/drive/MyDrive/QML/validation_parkinson/"

# Subjects Aggregation

In [ ]:
import os
import pandas as pd

# Base folder where GaCo and GaPt are stored
base_path = "/content/drive/MyDrive/QML/validation_parkinson/"

# Mapping: Folder → Label
folders = {
    "GaCo": 0,  # Healthy
    "GaPt": 1   # Parkinson's
}


In [ ]:
import random

# Set seed for reproducibility
random.seed(42)

# Paths
control_path = os.path.join(base_path, "GaCo")
patient_path = os.path.join(base_path, "GaPt")

# File lists
control_files = sorted([f for f in os.listdir(control_path) if f.endswith('.txt')])
patient_files = sorted([f for f in os.listdir(patient_path) if f.endswith('.txt')])

# Match the number of patients to the number of controls
balanced_patient_files = random.sample(patient_files, len(control_files))

# File mapping for balanced loading
balanced_files = {
    "GaCo": control_files,
    "GaPt": balanced_patient_files
}

In [ ]:
def load_gait_data_balanced(base_path, file_dict):
    all_data = []

    for folder, file_list in file_dict.items():
        label = 0 if folder == "GaCo" else 1
        folder_path = os.path.join(base_path, folder)

        for file in file_list:
            file_path = os.path.join(folder_path, file)
            try:
                df = pd.read_csv(file_path, delim_whitespace=True, header=None)
                df.columns = [f'ForcePlate_{i+1}' for i in range(df.shape[1])]
                df['Subject'] = f'{folder}_{file.split(".")[0]}'
                df['Label'] = label
                all_data.append(df)
            except Exception as e:
                print(f"❌ Failed to process {file_path}: {e}")

    return pd.concat(all_data, ignore_index=True)

# Load it
df_combined = load_gait_data_balanced(base_path, balanced_files)
print(f"✅ Loaded balanced dataset: {df_combined.shape}")
df_combined.head()

<ipython-input-4-2132be8ec0cd>:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None)
<ipython-input-4-2132be8ec0cd>:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None)
<ipython-input-4-2132be8ec0cd>:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None)
<ipython-input-4-2132be8ec0cd>:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None)
<ipython-input-4-2132be8ec0c

✅ Loaded balanced dataset: (912979, 21)


,ForcePlate_1,ForcePlate_2,ForcePlate_3,ForcePlate_4,ForcePlate_5,ForcePlate_6,ForcePlate_7,ForcePlate_8,ForcePlate_9,ForcePlate_10,...,ForcePlate_12,ForcePlate_13,ForcePlate_14,ForcePlate_15,ForcePlate_16,ForcePlate_17,ForcePlate_18,ForcePlate_19,Subject,Label
0,0.00,199.1,87.34,91.08,24.09,21.12,87.67,87.23,64.57,163.9,...,112.42,50.82,13.75,102.74,144.98,79.53,662.20,748.00,GaCo_GaCo01_01,0
1,0.01,199.1,87.34,91.08,24.09,21.12,87.67,87.23,64.57,163.9,...,112.42,50.82,13.75,102.74,144.98,79.53,662.20,748.00,GaCo_GaCo01_01,0
2,0.02,199.1,87.34,91.08,24.09,21.12,87.67,87.23,62.59,163.9,...,112.42,50.82,13.75,102.74,144.98,79.53,660.22,748.00,GaCo_GaCo01_01,0
3,0.03,199.1,87.34,91.08,24.09,21.12,87.67,89.10,64.57,163.9,...,112.42,48.07,13.75,105.49,144.98,79.53,664.07,745.69,GaCo_GaCo01_01,0
4,0.04,199.1,87.34,91.08,24.09,21.12,87.67,87.23,62.59,163.9,...,112.42,50.82,13.75,105.49,144.98,79.53,660.22,748.44,GaCo_GaCo01_01,0


In [ ]:
output_path = "/content/drive/MyDrive/QML/validation_parkinson/gaitpdb_features_raw.csv"
df_combined.to_csv(output_path, index=False)
print(f"✅ GaitPDB feature dataset saved to: {output_path}")

✅ GaitPDB feature dataset saved to: /content/drive/MyDrive/QML/validation_parkinson/gaitpdb_features_raw.csv


# Feature Extraction

In [ ]:
import numpy as np

In [ ]:
time_column = 'ForcePlate_1'
feature_columns = [col for col in df_combined.columns if col.startswith('ForcePlate_') and col != time_column]

In [ ]:
from scipy.stats import skew, kurtosis
from scipy.fft import fft
import numpy as np

def extract_statistical_features(window, feature_cols):
    features = {}
    for i, col in enumerate(feature_cols):
        data = window[:, i]

        # Time-domain features
        features[f"{col}_mean"] = np.mean(data)
        features[f"{col}_std"] = np.std(data)
        features[f"{col}_max"] = np.max(data)
        features[f"{col}_range"] = np.ptp(data)
        features[f"{col}_iqr"] = np.percentile(data, 75) - np.percentile(data, 25)
        features[f"{col}_energy"] = np.sum(np.square(data))
        features[f"{col}_rms"] = np.sqrt(np.mean(np.square(data)))
        features[f"{col}_skew"] = skew(data)
        features[f"{col}_kurtosis"] = kurtosis(data)

        # Frequency-domain features (basic FFT)
        fft_vals = np.abs(fft(data))
        fft_freqs = np.fft.fftfreq(len(data))

        # Use only positive frequencies
        pos_mask = fft_freqs > 0
        fft_vals = fft_vals[pos_mask]
        fft_freqs = fft_freqs[pos_mask]

        if len(fft_vals) > 0:
            dominant_freq = fft_freqs[np.argmax(fft_vals)]
        else:
            dominant_freq = 0.0

        features[f"{col}_dominant_freq"] = dominant_freq

    return features

In [ ]:
def apply_statistical_windowing(df, feature_cols, window_size=300, step_size=150):
    feature_rows = []
    labels = []
    subjects = []

    for subject, group in df.groupby("Subject"):
        data = group[feature_cols].values
        label = group["Label"].iloc[0]

        for start in range(0, len(data) - window_size + 1, step_size):
            end = start + window_size
            window = data[start:end]
            if window.shape[0] == window_size:
                features = extract_statistical_features(window, feature_cols)
                feature_rows.append(features)
                labels.append(label)
                subjects.append(subject)

    df_windowed = pd.DataFrame(feature_rows)
    df_windowed["Label"] = labels
    df_windowed["Subject"] = subjects

    return df_windowed

In [ ]:
df_windowed = apply_statistical_windowing(df_combined, feature_columns)
print(f"✅ Windowed dataset: {df_windowed.shape}")
df_windowed.head()

<ipython-input-39-6a1551e215fc>:18: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[f"{col}_skew"] = skew(data)
<ipython-input-39-6a1551e215fc>:19: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[f"{col}_kurtosis"] = kurtosis(data)


✅ Windowed dataset: (5951, 182)


,ForcePlate_2_mean,ForcePlate_2_std,ForcePlate_2_max,ForcePlate_2_range,ForcePlate_2_iqr,ForcePlate_2_energy,ForcePlate_2_rms,ForcePlate_2_skew,ForcePlate_2_kurtosis,ForcePlate_2_dominant_freq,...,ForcePlate_19_max,ForcePlate_19_range,ForcePlate_19_iqr,ForcePlate_19_energy,ForcePlate_19_rms,ForcePlate_19_skew,ForcePlate_19_kurtosis,ForcePlate_19_dominant_freq,Label,Subject
0,172.359367,143.888911,418.88,414.26,284.6800,1.512353e+07,224.525656,0.148185,-1.499030,0.006667,...,1131.79,1131.79,978.0650,1.741415e+08,761.886827,-0.449844,-1.370928,0.006667,0,GaCo_GaCo01_01
1,145.777133,152.755170,432.41,432.41,274.0925,1.337553e+07,211.151876,0.515931,-1.335367,0.006667,...,1158.30,1158.30,1000.9450,1.428210e+08,689.978327,0.136567,-1.853274,0.006667,0,GaCo_GaCo01_01
2,114.031500,136.622165,432.41,432.41,236.9400,9.500640e+06,177.957295,0.821438,-0.837736,0.006667,...,1220.45,1220.45,1012.1375,1.532822e+08,714.801181,0.071004,-1.834301,0.006667,0,GaCo_GaCo01_01
3,101.729833,132.147795,368.17,368.17,231.7700,8.343600e+06,166.769298,0.869513,-0.937864,0.006667,...,1220.45,1220.45,1031.6075,1.704292e+08,753.722360,-0.172900,-1.820402,0.006667,0,GaCo_GaCo01_01
4,96.336533,127.534214,376.09,376.09,213.6200,7.663711e+06,159.830233,0.911398,-0.770219,0.006667,...,1115.95,1115.95,1029.5450,1.849173e+08,785.105778,-0.413489,-1.718276,0.006667,0,GaCo_GaCo01_01


In [ ]:
output_path = "/content/drive/MyDrive/QML/validation_parkinson/gaitpdb_features.csv"
df_windowed.to_csv(output_path, index=False)
print(f"✅ GaitPDB feature dataset saved to: {output_path}")

✅ GaitPDB feature dataset saved to: /content/drive/MyDrive/QML/validation_parkinson/gaitpdb_features.csv
